### 초기설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi
%pip install sae-lens transformer-lens
%pip install neuronpedia
%pip uninstall numpy -y
%pip install "numpy==1.26.4"
%pip install --force-reinstall transformers

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thu Apr 16 11:33:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|          

### Gemma 모델 로드

In [3]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 토큰 파일 경로
file_path = '/content/drive/MyDrive/colab_ModelLoad.txt'
model_name = "google/gemma-2-9b-it"

if os.path.exists(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        hf_token = f.read().strip()
    print("성공적으로 토큰을 불러왔습니다.")
else:
    assert(f"오류: '{file_path}' 경로에 파일이 없습니다. 경로를 다시 확인해주세요.")

from huggingface_hub import login
login(token=hf_token)

device = "cuda" if torch.cuda.is_available() else "cpu"

# 토크나이저와 모델 로드 (token 옵션 추가 가능)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto"
)
model.eval()

print("Gemma 2 로드 완료!")

성공적으로 토큰을 불러왔습니다.


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Gemma 2 로드 완료!


### layers 변수 지정

In [4]:
if hasattr(model, "model") and hasattr(model.model, "layers"):
    layers = model.model.layers
elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
    layers = model.transformer.h
elif hasattr(model, "transformer") and hasattr(model.transformer, "layers"):
    layers = model.transformer.layers
else:
    print(model)
    raise AttributeError("이 모델의 레이어 구조(layers, h, blocks 등)를 찾을 수 없습니다.")

print(f"Model loaded: {model_name}")
print(f"Total layers: {len(layers)}")

Model loaded: google/gemma-2-9b-it
Total layers: 42


### SAE 로드

In [7]:
import os
import torch
import re
from transformers import AutoModelForCausalLM
from sae_lens import SAE
from neuronpedia.np_sae_feature import SAEFeature

SAE_LAYER = 30

sae, cfg_dict, sparsity = SAE.from_pretrained(
    release="gemma-scope-9b-pt-res-canonical",
    sae_id=f"layer_{SAE_LAYER}/width_16k/canonical",
)
sae = sae.to(model.device)
print(f"SAE 로드 완료 | layer {SAE_LAYER} | d_sae: {sae.cfg.d_sae}")

layer_30/width_16k/average_l0_120/params(…):   0%|          | 0.00/470M [00:00<?, ?B/s]

SAE 로드 완료 | layer 30 | d_sae: 16384


/tmp/ipykernel_4530/4263254917.py:10: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, cfg_dict, sparsity = SAE.from_pretrained(


### Load from Neuronpedia

In [ ]:
# Neuronpedia feature 확인 방법:
#   https://www.neuronpedia.org/gemma-2-9b/{layer}-gemmascope-res-16k/{feature_id}
#   예시: https://www.neuronpedia.org/gemma-2-9b/20-gemmascope-res-16k/12345

NP_MODEL_ID = "gemma-2-9b"
NP_SOURCE   = f"{SAE_LAYER}-gemmascope-res-16k"

def get_feature_info(feature_id: int, layer: int = SAE_LAYER):
    """
    Neuronpedia에서 feature 설명 및 top activation 텍스트 조회.
    확인용으로만 사용 — 실제 decoder 벡터는 SAE에서 직접 가져옴.
    """
    try:
        # model_id: gemma-2-9b, source: {layer}-gemmascope-res-16k
        feature = SAEFeature.get(
            model_id="gemma-2-9b",
            source=f"{layer}-gemmascope-res-16k",
            index=str(feature_id),
        )
        print(f"\n[Feature {feature_id}]")
        print(f"  URL: https://www.neuronpedia.org/{NP_MODEL_ID}/{NP_SOURCE}/{feature_id}")
        if hasattr(feature, "explanations") and feature.explanations:
            print(f"  설명: {feature.explanations}")
        if hasattr(feature, "activations") and feature.activations:
            print(f"  Top activations (최대 3개):")
            for act in feature.activations[:3]:
                print(f"    {act}")
    except Exception as e:
        print(f"[Feature {feature_id}] 조회 실패: {e}")
        print(f"  직접 확인: https://www.neuronpedia.org/{NP_MODEL_ID}/{NP_SOURCE}/{feature_id}")


def get_decoder_vector(feature_id: int):
    """SAE decoder에서 feature 방향 벡터 추출 후 정규화."""

    vec = sae.W_dec[feature_id].detach().clone().float()
    norm = vec.norm()
    vec_normalized = vec / (norm + 1e-8)
    
    return vec_normalized


### Feature 정의

In [ ]:
FEATURES = {
    "flower": [
        2712,   # flower
        15524,   # buy
        14898,  # +
        594,    # +=  
        2525,   # add
        
    ],
    "dollar": [
        1834,  # $
    ],
}

# 입력한 feature들 정보 확인
print("\n=== Feature 정보 확인 ===")
for concept, feat_ids in FEATURES.items():
    print(f"\n[{concept}]")
    for fid in feat_ids:
        get_feature_info(fid)

# 문제정의
from datasets import load_dataset

ds = load_dataset("lighteval/MATH", split="test")

# Level 3~4, algebra만 필터
filtered = ds.filter(
    lambda x: x["level"] in ["Level 3", "Level 4"]
    and x["type"] == "algebra"
)

# 랜덤 샘플 10개 출력
import random
samples = random.sample(range(len(filtered)), 10)

for i in samples:
    row = filtered[i]
    print(f"{'='*60}")
    print(f"[Level: {row['level']} | Type: {row['type']}]")
    print(f"Problem:\n{row['problem']}")
    print(f"\nSolution:\n{row['solution'][:300]}...")
    print()

PROBLEMS = [
    {
        "id": "gsm8k_1",
        "question": "Mary has 3 flowers. She buys 5 more flowers. How many flowers does she have now?",
        "answer": "8",
        "concept": "flower",
    },
]

PROMPT_TEMPLATE = """\
Solve the following problem step by step.
After your reasoning, write your final answer as 'Answer: [number]'.

Problem: {question}
Solution:"""


=== Feature 정보 확인 ===

[flower]
[Feature 2712] 조회 실패: No API key provided. Use one of these methods:
1. Context manager: with neuronpedia.api_key('key'): ...
2. Global setting: neuronpedia.set_api_key('key')
3. Parameter: FeatureRequest(api_key='key')
4. Environment: set NEURONPEDIA_API_KEY variable
  직접 확인: https://www.neuronpedia.org/gemma-2-9b/30-gemmascope-res-16k/2712
[Feature 15524] 조회 실패: No API key provided. Use one of these methods:
1. Context manager: with neuronpedia.api_key('key'): ...
2. Global setting: neuronpedia.set_api_key('key')
3. Parameter: FeatureRequest(api_key='key')
4. Environment: set NEURONPEDIA_API_KEY variable
  직접 확인: https://www.neuronpedia.org/gemma-2-9b/30-gemmascope-res-16k/15524
[Feature 14898] 조회 실패: No API key provided. Use one of these methods:
1. Context manager: with neuronpedia.api_key('key'): ...
2. Global setting: neuronpedia.set_api_key('key')
3. Parameter: FeatureRequest(api_key='key')
4. Environment: set NEURONPEDIA_API_KEY variable
  직접 확인

### Steering 정의

In [32]:
def make_steering_hook(decoder_vec, alpha, prompt_len):
    """
    CoT 생성 구간(prompt 이후) 전체에 steering 적용.
    alpha < 0: 개념 억제 / alpha > 0: 개념 강화
    """
    vec = decoder_vec.to(model.dtype)

    def hook_fn(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        hidden = hidden.clone()

        # autoregressive 생성: 현재 토큰(마지막 위치)이 CoT 구간이면 개입
        current_len = hidden.size(1)
        hidden[:, -1, :] += alpha * vec.to(hidden.device)

        return (hidden,) if isinstance(output, tuple) else hidden
    return hook_fn


### 답변 정의

In [30]:
def extract_answer(text):
    match = re.search(r"Answer:\s*(\d+)", text)
    if match:
        return match.group(1)
    numbers = re.findall(r"\d+", text)
    return numbers[-1] if numbers else "N/A"

def run_single(prompt_ids, feature_id, alpha, max_new_tokens=200):
    """feature_id의 decoder 벡터로 steering 후 생성."""
    prompt_len = prompt_ids.size(1)
    dec_vec = get_decoder_vector(feature_id)

    handle = layers[SAE_LAYER].register_forward_hook(
        make_steering_hook(dec_vec, alpha, prompt_len)
    )
    try:
        with torch.no_grad():
            out = model.generate(
                prompt_ids,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        handle.remove()

    gen_ids = out[0][prompt_len:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)


def run_baseline(prompt_ids, max_new_tokens=200):
    """steering 없이 생성."""
    with torch.no_grad():
        out = model.generate(
            prompt_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen_ids = out[0][prompt_ids.size(1):]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)


### 전체 실험

In [35]:
ALPHAS = [-200, -100, 0, 100, 200]
MAX_NEW_TOKENS = 250

print("\n" + "=" * 70)
print(f"모델: {model_name} | SAE layer: {SAE_LAYER}")
print("=" * 70)

for problem in PROBLEMS:
    concept = problem["concept"]
    feat_ids = FEATURES.get(concept, [])

    if not feat_ids:
        print(f"\n[{problem['id']}] feature ID가 없음")
        continue

    print(f"\n{'='*70}")
    print(f"[{problem['id']}] {problem['question']}")
    print(f"정답: {problem['answer']} | 개념: {concept} | features: {feat_ids}")
    print("=" * 70)

    prompt = PROMPT_TEMPLATE.format(question=problem["question"])
    prompt_ids = tokenizer.encode(
        prompt, return_tensors="pt"
    ).to(model.device)

    # Baseline
    baseline_text = run_baseline(prompt_ids, MAX_NEW_TOKENS)
    baseline_ans = extract_answer(baseline_text)
    correct = "✓" if baseline_ans == problem["answer"] else "✗"
    print(f"\n[BASELINE]")
    print(f"  답변: {baseline_ans} {correct}")
    print(f"  CoT:\n{baseline_text[:300]}")

    # Feature별 steering
    for feat_id in feat_ids:
        print(f"\n--- feature {feat_id} ---")
        for alpha in ALPHAS:
            if alpha == 0:
                continue  # baseline과 동일

            text = run_single(prompt_ids, feat_id, alpha, MAX_NEW_TOKENS)
            ans = extract_answer(text)
            correct = "✓" if ans == problem["answer"] else "✗"
            changed = "★ 답 바뀜" if ans != baseline_ans else ""

            print(f"  CoT:\n{text[:300]}")
            print(f"  alpha={alpha:+4d} | 답변: {ans} {correct} {changed}")
            if changed:
                print(f"    CoT: {text[:200]}...")

print("\n실험 완료")


모델: google/gemma-2-9b-it | SAE layer: 30

[gsm8k_1] Mary has 3 flowers. She buys 5 more flowers. How many flowers does she have now?
정답: 8 | 개념: flower | features: [2712, 15524, 14898, 594, 2525]

[BASELINE]
  답변: 8 ✓
  CoT:

1. **Start with the initial number of flowers:** Mary begins with 3 flowers.
2. **Add the number of flowers she buys:** She buys 5 more flowers.
3. **Calculate the total:** 3 + 5 = 8

Answer: 8 


--- feature 2712 ---
  CoT:

1. Mary starts with 3 flowers.
2. She buys 5 more flowers.
3. To find the total number of flowers, we add the starting number (3) and the number she bought (5): 3 + 5 = 8

Answer: 8 

  alpha=-200 | 답변: 8 ✓ 
  CoT:

1. **Start with the initial number of flowers:** Mary begins with 3 flowers.
2. **Add the new flowers:** She buys 5 more flowers.
3. **Calculate the total:** 3 + 5 = 8

Answer: 8 

  alpha=-100 | 답변: 8 ✓ 
  CoT:

1. **Start with the initial number of flowers:** Mary begins with 3 flowers.
2. **Add the number of flowers she buys:*